### Run Ollama LLM as a Local LLM on Google Colab


In [1]:
!pip install pyngrok==7.2.0

In [ ]:
import os
import subprocess
import time
import requests
from google.colab import userdata
from pyngrok import ngrok, conf

# =====================================================================
# 1. INSTALL OLLAMA AND UTILITIES
# =====================================================================
print("--- 1. Installing zstd and Ollama ---")
subprocess.run("sudo apt-get update && sudo apt-get install -y zstd", shell=True, stdout=subprocess.DEVNULL)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, stdout=subprocess.DEVNULL)

# =====================================================================
# 2. LAUNCH OLLAMA DAEMON IN BACKGROUND
# =====================================================================
print("--- 2. Starting Ollama Server on Port 1234 ---")
# Kill any prior instances of ollama to prevent conflicts
subprocess.run("pkill -f ollama", shell=True)
time.sleep(2)

# Serve Ollama using custom host and origins
subprocess.Popen(
    "OLLAMA_HOST=0.0.0.0:1234 OLLAMA_ORIGIN=* nohup ollama serve > /content/nohup.out 2>&1 &",
    shell=True
)
time.sleep(5)  # Wait for startup

# =====================================================================
# 3. PULL SPECIFIED MODELS
# =====================================================================
gen_model = "gemma2:9b-instruct-q5_K_S"
embed_model = "nomic-embed-text"

print(f"--- 3. Pulling Generation Model: {gen_model} ---")
subprocess.run(f"OLLAMA_HOST=127.0.0.1:1234 ollama pull {gen_model}", shell=True)

print(f"--- 3. Pulling Embedding Model: {embed_model} ---")
subprocess.run(f"OLLAMA_HOST=127.0.0.1:1234 ollama pull {embed_model}", shell=True)

# =====================================================================
# 4. INITIALIZE NGROK PUBLIC TUNNEL
# =====================================================================
print("--- 4. Establishing ngrok Tunnel ---")
ngrok_auth = userdata.get('NGROK_AUTH_TOKEN')
conf.get_default().auth_token = ngrok_auth

# Disconnect existing tunnels if any
ngrok.kill()
time.sleep(1)

public_url = ngrok.connect('1234').public_url
print(f"\nYour Live Model Base URL is: {public_url}\n")

# =====================================================================
# 5. DIAGNOSTIC API TESTS
# =====================================================================
print("--- 5. Verifying Active Endpoints ---")

try:
    # Test Generation
    gen_response = requests.post(
        f"{public_url}/api/chat",
        json={
            "model": gen_model,
            "stream": False,
            "messages": [{"role": "user", "content": "What is the capital of France?"}]
        }
    )
    print("Generation Test: OK")
    print("Response:", gen_response.json().get('message', {}).get('content'))
except Exception as e:
    print("Generation Test Failed:", e)

try:
    # Test Embeddings
    embed_response = requests.post(
        f"{public_url}/api/embeddings",
        json={
            "model": embed_model,
            "prompt": "Paris"
        }
    )
    dim_size = len(embed_response.json().get('embedding', []))
    print(f"Embedding Test: OK (Dimensions: {dim_size})")
except Exception as e:
    print("Embedding Test Failed:", e)

--- 1. Installing zstd and Ollama ---
--- 2. Starting Ollama Server on Port 1234 ---
--- 3. Pulling Generation Model: gemma2:9b-instruct-q5_K_S ---
--- 3. Pulling Embedding Model: nomic-embed-text ---
--- 4. Establishing ngrok Tunnel ---

Your Live Model Base URL is: https://blabber-greedless-tumble.ngrok-free.dev

--- 5. Verifying Active Endpoints ---
Generation Test: OK
Response: The capital of France is **Paris**. 🇫🇷  

Embedding Test: OK (Dimensions: 768)


### How to Use Your Model Externally

You can connect to your running model from any other application (Python, JavaScript, Mobile, etc.) using your public ngrok URL.

#### Option A: Direct API Call (Python Example)
```python
import requests

base_url = "https://blabber-greedless-tumble.ngrok-free.dev"

payload = {
    "model": "gemma2:9b-instruct-q5_K_S",
    "messages": [
        {"role": "user", "content": "Hello! How are you?"}
    ],
    "stream": False
}

response = requests.post(f"{base_url}/api/chat", json=payload)
print(response.json()['message']['content'])
```

#### Option B: Using OpenAI Compatibility Endpoint
Ollama provides an OpenAI-compatible API layer out-of-the-box. You can point the official `openai` SDK directly to your ngrok URL:

```python
from openai import OpenAI

client = OpenAI(
    base_url="https://blabber-greedless-tumble.ngrok-free.dev/v1",
    api_key="ollama" # Required but ignored by Ollama
)

response = client.chat.completions.create(
    model="gemma2:9b-instruct-q5_K_S",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)
```

---
